# Pipeline Bronze → Silver — Transformações e Decisões de Modelagem

Demonstra, lado a lado, o que muda entre a camada Bronze (dado bruto) e a
camada Silver (limpo, decodificado e integrado), e evidencia por que algumas
decisões de modelagem foram necessárias — em especial a separação entre o
resultado "headline" de um município e o resultado usado para comparar com a
meta municipal.

Pré-requisito: rodar antes `python pipeline/batch/01_ingestao_bronze.py` e
`python pipeline/batch/02_processamento_silver.py` a partir da raiz do projeto
(ou executar as células deste notebook, que chamam as mesmas funções).

In [1]:
import sys
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

sys.path.insert(0, str(Path.cwd().parent))
from pipeline.batch.config import BRONZE_DIR, SILVER_DIR, REDE_MAP

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

def ler(camada_dir, nome):
    return pq.ParquetDataset(str(camada_dir / nome)).read().to_pandas()


## 1. Decodificação de `rede` — código numérico → rótulo

In [2]:
bronze_mun = ler(BRONZE_DIR, "indicador_municipio")
silver_mun = ler(SILVER_DIR, "indicador_municipio")

print("Bronze — 'rede' como veio da fonte:")
print(bronze_mun[["ano", "id_municipio", "rede", "taxa_alfabetizacao"]].head(4))
print()
print("Silver — 'rede' decodificado + flag de validação:")
print(silver_mun[["ano", "id_municipio", "rede", "rede_label", "taxa_alfabetizacao", "flag_taxa_invalida"]].head(4))
print()
print("REDE_MAP usado (inferido — dicionário oficial do BigQuery não acessível sem credenciais):")
print(REDE_MAP)

Bronze — 'rede' como veio da fonte:
    ano id_municipio  rede  taxa_alfabetizacao
0  2023      1100031     3               69.10
1  2023      1100072     3               58.20
2  2023      1100189     5               69.73
3  2023      1101609     3               50.70

Silver — 'rede' decodificado + flag de validação:
    ano id_municipio  rede rede_label  taxa_alfabetizacao  flag_taxa_invalida
0  2023      1100031     3  Municipal               69.10               False
1  2023      1100072     3  Municipal               58.20               False
2  2023      1100189     5    Pública               69.73               False
3  2023      1101609     3  Municipal               50.70               False

REDE_MAP usado (inferido — dicionário oficial do BigQuery não acessível sem credenciais):
{0: 'Total', 1: 'Federal', 2: 'Estadual', 3: 'Municipal', 4: 'Privada', 5: 'Pública'}


## 2. Despivotagem das metas (wide → long)

In [3]:
bronze_meta = ler(BRONZE_DIR, "meta_municipio")
silver_meta = ler(SILVER_DIR, "meta_municipio")

exemplo_municipio = "1100031"

print(f"Bronze — meta_municipio para id_municipio={exemplo_municipio} (formato largo, uma linha por vintage):")
cols_meta = [c for c in bronze_meta.columns if c.startswith("meta_alfabetizacao_") or c in ("ano", "id_municipio", "rede")]
print(bronze_meta[bronze_meta["id_municipio"] == exemplo_municipio][cols_meta])
print()
print(f"Silver — mesmo município em formato long (uma linha por ano_meta, só a vintage mais recente):")
print(silver_meta[silver_meta["id_municipio"] == exemplo_municipio].sort_values("ano_meta"))

Bronze — meta_municipio para id_municipio=1100031 (formato largo, uma linha por vintage):
     id_municipio       rede  meta_alfabetizacao_2024  meta_alfabetizacao_2025  meta_alfabetizacao_2026  meta_alfabetizacao_2027  meta_alfabetizacao_2028  \
3491      1100031  Municipal                    70.85                    72.53                    74.15                    75.71                    77.21   
8843      1100031  Municipal                    70.85                    72.53                    74.15                    75.71                    77.21   

      meta_alfabetizacao_2029  meta_alfabetizacao_2030   ano  
3491                    78.64                       80  2023  
8843                    78.64                       80  2024  

Silver — mesmo município em formato long (uma linha por ano_meta, só a vintage mais recente):
      id_municipio       rede  taxa_alfabetizacao  nivel_alfabetizacao  percentual_participacao  ano_referencia  valor_meta            dt_processamento  \

A vintage de 2024 revisou a trajetória publicada em 2023 — para cada `ano_meta`
mantemos apenas o valor da vintage mais recente (`ano_referencia` mais alto),
evitando que a mesma meta apareça duplicada com dois valores diferentes.

## 3. Por que o resultado "headline" não pode ser comparado direto com a meta municipal

In [4]:
# meta_municipio só define metas para a rede "Municipal". Um município pode ter
# mais de uma rede avaliada (Municipal, Estadual, Pública=agregado). O headline
# escolhe a melhor rede disponível (prioridade Pública > Municipal > ...) — que
# pode ser diferente da rede "Municipal" usada pela meta.
caso = silver_mun[(silver_mun["id_municipio"] == "3501608") & (silver_mun["ano"] == 2024)]
print("Todas as redes avaliadas para Americana/SP em 2024:")
print(caso[["rede_label", "taxa_alfabetizacao"]])

Todas as redes avaliadas para Americana/SP em 2024:
      rede_label  taxa_alfabetizacao
20820  Municipal               64.36
21351    Pública               64.87
21794   Estadual               65.38


In [5]:
integrado = ler(SILVER_DIR, "alfabetizacao_integrado")
linha = integrado[(integrado["id_municipio"] == "3501608") & (integrado["ano"] == 2024)]
print("Linha final na alfabetizacao_integrado:")
print(linha[["id_municipio", "nome_municipio", "sigla_uf",
             "rede_label", "taxa_alfabetizacao",
             "taxa_alfabetizacao_municipal", "meta_alfabetizacao_municipal",
             "gap_meta_municipal", "atingiu_meta_municipal"]].to_string(index=False))

Linha final na alfabetizacao_integrado:
id_municipio nome_municipio sigla_uf rede_label  taxa_alfabetizacao  taxa_alfabetizacao_municipal  meta_alfabetizacao_municipal  gap_meta_municipal atingiu_meta_municipal
     3501608      Americana       SP    Pública               64.87                         64.36                         62.11                2.25                   True


`taxa_alfabetizacao` (headline, rede "Pública" = 64.87) e
`taxa_alfabetizacao_municipal` (rede "Municipal" = 64.36) são diferentes.
Se a comparação com a meta usasse o headline em vez do valor específico da
rede Municipal, o gap ficaria levemente errado (mistura de escopos). É por
isso que `integrar_bases()` em `02_processamento_silver.py` calcula
`gap_meta_municipal` a partir de `taxa_alfabetizacao_municipal`, não do
headline.

## 4. Integração completa — diretório de municípios entra via join

In [6]:
print(f"{len(integrado):,} registros — um por (ano, id_município)")
integrado[["ano", "id_municipio", "nome_municipio", "sigla_uf", "nome_uf", "nome_regiao",
           "rede_label", "taxa_alfabetizacao"]].sample(5, random_state=7)

11,030 registros — um por (ano, id_município)


,ano,id_municipio,nome_municipio,sigla_uf,nome_uf,nome_regiao,rede_label,taxa_alfabetizacao
1195,2023,2602902,Cabo de Santo Agostinho,PE,Pernambuco,Nordeste,Pública,55.12
10117,2024,2926103,Retirolândia,BA,Bahia,Nordeste,Pública,23.58
900,2023,3144375,Natalândia,MG,Minas Gerais,Sudeste,Pública,49.23
4233,2023,5003256,Costa Rica,MS,Mato Grosso do Sul,Centro-Oeste,Pública,38.85
10049,2024,3505005,Barão de Antonina,SP,São Paulo,Sudeste,Pública,64.23


## 5. Qualidade de dados — Bronze e Silver

In [7]:
from quality.validacao_dados import validar_camada

resultado_bronze = validar_camada("bronze")
resultado_silver = validar_camada("silver")

for camada, resultado in [("bronze", resultado_bronze), ("silver", resultado_silver)]:
    ok = sum(1 for r in resultado.values() if r["status"] == "OK")
    print(f"{camada}: {ok}/{len(resultado)} tabelas OK")

2026-08-16 16:56:14 | INFO     | quality.validacao_dados | ============================================================


2026-08-16 16:56:14 | INFO     | quality.validacao_dados | VALIDAÇÃO — CAMADA BRONZE — 2026-08-16 16:56:14


2026-08-16 16:56:14 | INFO     | quality.validacao_dados | ============================================================


2026-08-16 16:56:14 | INFO     | quality.validacao_dados |   [indicador_municipio] OK — 23,995 registros


2026-08-16 16:56:14 | INFO     | quality.validacao_dados |   [indicador_uf] OK — 145 registros


...

2026-08-16 16:56:24 | INFO     | quality.validacao_dados | ────────────────────────────────────────────────────────────


2026-08-16 16:56:24 | INFO     | quality.validacao_dados |   Total de alertas: 0


2026-08-16 16:56:24 | INFO     | quality.validacao_dados |   Tabelas OK: 10/10


2026-08-16 16:56:24 | INFO     | quality.validacao_dados | ============================================================


bronze: 8/8 tabelas OK
silver: 10/10 tabelas OK


## Conclusões

- A Silver decodifica `rede`, despivota as metas para formato long (mantendo só
  a vintage mais recente por ano-alvo) e integra resultado + diretório + meta
  em uma linha por (ano, município).
- A decisão de manter `taxa_alfabetizacao` (headline) separado de
  `taxa_alfabetizacao_municipal` (escopo da meta) não é cosmética — comparar
  o headline com a meta municipal produziria um gap sistematicamente errado
  para qualquer município com mais de uma rede avaliada.
- Todas as 18 tabelas Bronze+Silver passam nas regras de qualidade definidas em
  `quality/validacao_dados.py` (completude, unicidade, integridade referencial
  e domínio de UF).